# End-To-End Melody Pipeline

Load one audio file from `experiments/audio_samples`, extract the harmonic audio, set one global pitch range, optionally strip frequencies below the lowest note, run every registered PitchLab model, sonify each model result immediately, then optionally save CSVs to `experiments/pitch_runs`.

## 1. Setup

In [ ]:
from pathlib import Path
import sys

import librosa

try:
    from IPython.display import Audio, Markdown, display
except ModuleNotFoundError:
    Audio = None
    Markdown = str

    def display(value):
        print(value)

PROJECT_ROOT = Path.cwd()
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"
AUDIO_SAMPLES_DIR = EXPERIMENTS_DIR / "audio_samples"
PITCH_RUNS_DIR = EXPERIMENTS_DIR / "pitch_runs"
PITCHLAB_DIR = PROJECT_ROOT / "melody_extractor" / "pitchlab-ernie"
REFINER_DIR = PROJECT_ROOT / "melody_refiner"

for path in (PITCHLAB_DIR, REFINER_DIR):
    if path.exists():
        sys.path.insert(0, str(path))

import pitchlab
from pitchlab.sonify import sonify_f0_dataframe

import melody_extractor.tools as extractor_tools
import melody_refiner.tools as refiner_tools

TIME_COLUMN = "time"
FREQUENCY_COLUMN = "frequency_hz"
MODEL_COLUMN = "model"
SONIFICATION_SAMPLE_RATE = 44100

audio_name = "uzbek_dari.mp3"
audio_samples_path = AUDIO_SAMPLES_DIR / audio_name

if not audio_samples_path.exists():
    available_audio = ", ".join(path.name for path in sorted(AUDIO_SAMPLES_DIR.glob("*")))
    raise FileNotFoundError(
        f"Could not find {audio_samples_path}. Available audio samples: {available_audio}"
    )

## 2. Load And Preview Audio

In [ ]:
raw_audio, sr = extractor_tools.load_audio(audio_samples_path)

if Audio is not None:
    pass
    #display(Audio(filename=audio_samples_path))

## 3. Extract Harmonic Audio

In [ ]:
harmonic_audio = extractor_tools.extract_harmonic_part(raw_audio)
extractor_tools.show_audio(harmonic_audio, sr, "Extracted Harmonic Part")

## 4. Optional Sine-Wave Pitch Finder

In [ ]:
from extra_stuff import sound_slider
sound_slider()

## Pitch Range And Optional Low-Cut

In [ ]:
def note_or_hz_to_hz(value):
    value = value.strip()
    if not value:
        raise ValueError("A note name or frequency is required.")
    try:
        return float(value)
    except ValueError:
        try:
            return float(librosa.note_to_hz(value))
        except Exception as error:
            raise ValueError(f"Could not parse note or Hz value: {value!r}") from error

analysis_audio = harmonic_audio

#default values for skipping
LOWEST_HZ=26
HIGHEST_HZ = 2000
PITCH_RANGE_HZ = (LOWEST_HZ, HIGHEST_HZ)

quit = input("press q to skip:")
if quit.lower() != "q":
    lowest_note = input("Enter lowest note or Hz: ").strip()
    highest_note = input("Enter highest note or Hz: ").strip()
    
    LOWEST_HZ = note_or_hz_to_hz(lowest_note)
    HIGHEST_HZ = note_or_hz_to_hz(highest_note)
    PITCH_RANGE_HZ = (LOWEST_HZ, HIGHEST_HZ)
    
    if LOWEST_HZ >= HIGHEST_HZ:
        raise ValueError(f"Lowest note must be below highest note: {PITCH_RANGE_HZ}")
    
    strip_below_lowest = input(
        "Strip sound waves below the lowest note? [y/N]: "
    ).strip().lower() in {"y", "yes"}
    
    if strip_below_lowest:
        analysis_audio = extractor_tools.high_pass(harmonic_audio, sr, LOWEST_HZ, order=10)
        extractor_tools.show_audio(
            analysis_audio,
            sr,
            f"Harmonic Audio With Frequencies Below {LOWEST_HZ:.2f} Hz Removed",
        )
    
    print(f"pitch range: {LOWEST_HZ:.2f}-{HIGHEST_HZ:.2f} Hz")
    print(f"low-cut below lowest note: {strip_below_lowest}")

In [ ]:
analysis_section = {
    "number": 1,
    "start_seconds": 0.0,
    "end_seconds": len(analysis_audio) / sr,
    "audio": analysis_audio,
    "sr": sr,
}

## Run Every Pitch Model And Sonify

In [ ]:
import traceback
import warnings

#warnings are annoying
with warnings.catch_warnings():
    warnings.simplefilter('ignore')

    EXCEPT_MODELS = {"torchcrepe", "crepe", "yaapt"}
    #crepe is very old. yaapt is from the almost the 1990s
    
    MODEL_LIST = [model for model in pitchlab.available_models() if model not in EXCEPT_MODELS]
    
    pitch_runs = []
    failures = []
    
    for model_name in MODEL_LIST:
        print(pitchlab.print_description(model_name))
        print(f"\nrunning {model_name}")
        try:
            pitch_run = extractor_tools.run_pitch_model(
                analysis_section,
                model_name,
                PITCH_RANGE_HZ,
            ).copy()
            pitch_run["source_audio"] = audio_name
            pitch_run.attrs["source_stem"] = f"{Path(audio_name).stem}_{model_name}"
            pitch_runs.append(pitch_run)
            
            display(Markdown(f"### {model_name}"))
            sonified_audio = sonify_f0_dataframe(
                pitch_run,
                sample_rate=SONIFICATION_SAMPLE_RATE,
                frequency_column=FREQUENCY_COLUMN,
            )
            if len(sonified_audio) and Audio is not None:
                display(Audio(sonified_audio, rate=SONIFICATION_SAMPLE_RATE))
            else:
                print("  no sonified audio generated")
        except Exception as error:
            failures.append({"model": model_name, "error": error})
            print(f"  failed {model_name}: {error}")
            traceback.print_exc()
    
    print(f"\nin-memory pitch runs: {len(pitch_runs)}")
    print(f"failed model runs: {len(failures)}")
    
    refiner_tools.print_pitch_run_summary(
        pitch_runs,
        time_column=TIME_COLUMN,
        frequency_column=FREQUENCY_COLUMN,
    )
    fig, ax = refiner_tools.plot_pitch_runs(
        pitch_runs,
        time_column=TIME_COLUMN,
        frequency_column=FREQUENCY_COLUMN,
    )
    
    if failures:
        print("\nFailures:")
        for failure in failures:
            print(f"- {failure['model']}: {failure['error']}")

## Optional Final CSV Save

In [ ]:
save_csvs = input("Save model CSVs in experiments/pitch_runs? [y/N]: ").strip().lower() in {
    "y",
    "yes",
}

if save_csvs:
    if not pitch_runs:
        saved_paths = []
        print("No pitch runs to save.")
    else:
        saved_paths = refiner_tools.save_pitch_runs(
            pitch_runs,
            PITCH_RUNS_DIR,
            model_column=MODEL_COLUMN,
            suffix="",
        )
        for path in saved_paths:
            print(f"saved {path.relative_to(PROJECT_ROOT)}")
else:
    saved_paths = []
    print("CSV save skipped.")